In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from netgen.geom2d import SplineGeometry
import numpy as np
from math import sqrt
import inspect
"""
der Code berechnet den Abstand mittels L2-norm und den maximalem Fehler zweier Lösungen, mit unterschiedlicher Schrittweite maxh
"""

def skalar_L2norm_abstand(filename_ref,maxh, step, order, mass_Omega):
    """
    wir berechnen die L2norm über ein funktionenraster, dabei erhalten wir folgende formel
    ||uh-uref||_2 = sqrt( sum_{i=1}^{anzahl_punkte} (u_h(i)-u_ref(i))^2 * maß(omega)/anzahl_pkt)
    = sqrt( summe[ (uh(i)-uref(i))^2 ] * maß_Omega / anzahl_pkte )
    """
    filename_func = f"xyzFiles/h{maxh}_s{step}_o{order}_u.xyz"
    data_u_h = np.loadtxt(filename_func)
    #x_uh = data_u_h[:,0]    
    #y_uh = data_u_h[:,1]
    z_uh = data_u_h[:,2]

    data_u_ref = np.loadtxt(filename_ref)
    #x_u_ref = data_u_ref[:,0]    
    #y_u_ref = data_u_ref[:,1]
    z_u_ref = data_u_ref[:,2]

    normierungsZahl = step*step
    sum = 0
    maxDiff = 0

    for i in range(normierungsZahl):
        diff = abs(z_uh[i]-z_u_ref[i])
        #berechnen Maximalabstand
        if diff > maxDiff:
            maxDiff = diff

        #quadrierter summand für integral-summe
        sum += diff*diff
    integral = sum/normierungsZahl*mass_Omega
    L2_norm_h = sqrt(integral)
    # print("L2norm - done")
    return L2_norm_h, maxDiff

def tensor_L2norm_abstand(filename_ref,maxh, step, order, mass_Omega):
    """
    wir berechnen die L2norm über ein funktionenraster, dabei erhalten wir folgende formel
    ||uh-uref||_2 = sqrt( sum_{i=1}^{anzahl_punkte} (u_h(i)-u_ref(i))^2 * maß(omega)/anzahl_pkt)
    = sqrt( summe[ (uh(i)-uref(i))^2 ] * maß_Omega / anzahl_pkte )
    """
    filename_func = f"xyzFiles/h{maxh}_s{step}_o{order}_sigma.xyz"
    data_sigma_h = np.loadtxt(filename_func)
    #x_sigma_h = data_func[:,0]    
    #y_sigma_h = data_func[:,1]
    z_sigma_h_0 = data_sigma_h[:,2]
    z_sigma_h_1 = data_sigma_h[:,3]
    z_sigma_h_2 = data_sigma_h[:,4]
    

    data_sigma_ref = np.loadtxt(filename_ref)
    #x_sigma_ref = data_u_ref[:,0]    
    #y_sigma_ref = data_u_ref[:,1]
    z_sigma_ref_0 = data_sigma_ref[:,2]
    z_sigma_ref_1 = data_sigma_ref[:,3]
    z_sigma_ref_2 = data_sigma_ref[:,4]

    normierungsZahl = step*step
    sum = 0
    maxSigmaDiff = 0

    for i in range(normierungsZahl):
        diff_0 = (z_sigma_ref_0[i]-z_sigma_h_0[i])
        diff_1 = (z_sigma_ref_1[i]-z_sigma_h_1[i])
        diff_2 = (z_sigma_ref_2[i]-z_sigma_h_2[i])

        frobenius_diff = diff_0*diff_0 + 2*diff_1*diff_1 + diff_2*diff_2

        # diff = abs(z_uh[i]-z_u_ref[i])
        #berechnen Maximalabstand
        if frobenius_diff > maxSigmaDiff:
            maxSigmaDiff = frobenius_diff

        #quadrierter summand für integral-summe
        sum += frobenius_diff
    integral = sum*mass_Omega/normierungsZahl
    L2_norm_sigma = sqrt(integral)
    # print("L2norm - done")
    return L2_norm_sigma, maxSigmaDiff

if __name__ == "__main__":
    l = 200
    b = 100
    t = 5
    step = 40
    maxh = 100 # in {10, 50, 100, 200}
    order = 4
    mass_Omega = l*b
    print("gestartet!")
    print(f"l={l}, b={b}, t={t}, step={step}, maxh={maxh}, order={order}")
    # maxh_array = [1,5,10,50,100,200] 
    # maxh_array = [500,200,100,50,10,5]
    maxh_array = [5,10,50,100,200,500]

    # fehlerauswertung
    L2_u_array = np.zeros(6)
    maxDiff_u_array = np.zeros(6)
    L2_sigma_array = np.zeros(6)
    maxDiff_sigma_array = np.zeros(6)

    for index, maxh_h in enumerate(maxh_array):
        print("maxh = ",maxh_h)

        filename_u_ref = f"xyzFiles/h1_s40_o{order}_u.xyz"
        filename_sigma_ref = f"xyzFiles/h1_s40_o{order}_sigma.xyz"

        L2fehler_u, maxDiff_u = skalar_L2norm_abstand(filename_u_ref, maxh_h, step, order, mass_Omega)
        L2fehler_sigma, maxDiff_sigma = tensor_L2norm_abstand(filename_sigma_ref, maxh_h, step, order, mass_Omega)

        L2_u_array[index] = L2fehler_u
        maxDiff_u_array[index] = maxDiff_u
        L2_sigma_array[index] = L2fehler_sigma
        maxDiff_sigma_array[index] = maxDiff_sigma


        print(" - L2-fehler u: ",L2fehler_u,"\n - max Abstand u: ",maxDiff_u,"\n - L2-fehler sigma: ",L2fehler_sigma,"\n - max Abstand sigma: ",maxDiff_sigma)



gestartet!
l=200, b=100, t=5, step=40, maxh=100, order=4
maxh =  5
 - L2-fehler u:  0.0 
 - max Abstand u:  0 
 - L2-fehler sigma:  0.06836977142918851 
 - max Abstand sigma:  8.110840899022797e-05
maxh =  10
 - L2-fehler u:  5.000000000000376e-10 
 - max Abstand u:  1.0000000000000751e-10 
 - L2-fehler sigma:  0.30308850973682055 
 - max Abstand sigma:  0.0013669536709268071
maxh =  50
 - L2-fehler u:  6.045932516990504e-07 
 - max Abstand u:  1.7000000000003395e-08 
 - L2-fehler sigma:  17.75658137869262 
 - max Abstand sigma:  0.873383925359355
maxh =  100
 - L2-fehler u:  1.6515751375580814e-05 
 - max Abstand u:  4.3630000000000123e-07 
 - L2-fehler sigma:  93.69055704408201 
 - max Abstand sigma:  14.661240663968192
maxh =  200
 - L2-fehler u:  1.4634202314270465e-05 
 - max Abstand u:  3.9489999999999854e-07 
 - L2-fehler sigma:  87.68806997265064 
 - max Abstand sigma:  34.70027206062496
maxh =  500
 - L2-fehler u:  1.4634202314270465e-05 
 - max Abstand u:  3.9489999999999854e